In [1]:
# This Notebook converts soft label annotations to YOLO format for training.
# This data is then directly usable by Ultralytics models.

In [1]:
soft_label_thresh = 0.5

# Pascal VOC

In [2]:
import os
import json

existing = "/path/to/datasets/VOC"
new = "/path/to/datasets/VOC_ours"

if os.path.exists(new):
    print(f"Removing existing directory: {new}")
    os.system(f"rm -r {new}")

val_label_path = new + "/labels" + "/val2012"
train_label_path = new + "/labels" + "/train2012"

val_soft_label_path = val_label_path.replace("labels", "soft_labels")
train_soft_label_path = train_label_path.replace("labels", "soft_labels")

# create new dataset directory structure
os.makedirs(new, exist_ok=True)
os.makedirs(val_label_path, exist_ok=True)
os.makedirs(train_label_path, exist_ok=True)
os.makedirs(val_soft_label_path, exist_ok=True)
os.makedirs(train_soft_label_path, exist_ok=True)
# copy images via hardlink
os.system(f"cp -al {existing}/images {new}/images")

Removing existing directory: /path/to/datasets/VOC_ours


0

In [3]:
val_data = json.load(open("../../our_datasets/soft_PascalVOC_2012_detection_val.json"))

# filenames = list(val_data["objects"].keys())
filenames = os.listdir(val_label_path.replace("labels", "images")) # get list of image filenames from the images folder -> empty txt file if no labels
fname_to_img_dimensions = {img["file_name"]: (img["width"], img["height"]) for img in val_data["images"]}

# Create YOLO annotation files for validation set
for filename in filenames:

    if not filename in val_data["objects"]:
        # txt_filename = os.path.splitext(filename)[0] + ".txt"
        # txt_path = os.path.join(val_label_path, txt_filename)
        # open(txt_path, "w").close() # create empty txt file for images without annotations

        # txt_path = os.path.join(val_soft_label_path, txt_filename)
        # open(txt_path, "w").close() # create empty txt file for soft labels

        continue

    obj = val_data["objects"][filename] # annotations for this image (list of dicts)

    # create .txt file for this image
    txt_filename = os.path.splitext(filename)[0] + ".txt"
    txt_path = os.path.join(val_label_path, txt_filename)
    with open(txt_path, "w") as f:
        for ann in obj:
            soft_label = ann["postprocessed_soft_label"]
            if max(soft_label[:-1]) < soft_label_thresh:
                continue  # skip if no class has a soft label prob. above the threshold
            
            cls = soft_label.index(max(soft_label[:-1]))  # class with highest soft label probability (exclude background)

            w, h = fname_to_img_dimensions[filename]
            # Convert bbox to YOLO format (x_center, y_center, width, height)
            x_min, y_min, x_max, y_max = ann["bbox"]
            x_center = (x_min + x_max) / 2 / w
            y_center = (y_min + y_max) / 2 / h
            width = (x_max - x_min) / w
            height = (y_max - y_min) / h

            f.write(f"{cls} {x_center} {y_center} {width} {height}\n")

    # store soft label in separate folder 
    txt_path = os.path.join(val_soft_label_path, txt_filename)
    with open(txt_path, "w") as f:
        for ann in obj:
            soft_label = ann["postprocessed_soft_label"]
            if max(soft_label[:-1]) < soft_label_thresh:
                continue  # skip if no class has a soft label prob. above the threshold
            
            cls = soft_label.index(max(soft_label[:-1]))  # class with highest soft label probability (exclude background)

            w, h = fname_to_img_dimensions[filename]
            # Convert bbox to YOLO format (x_center, y_center, width, height)
            x_min, y_min, x_max, y_max = ann["bbox"]
            x_center = (x_min + x_max) / 2 / w
            y_center = (y_min + y_max) / 2 / h
            width = (x_max - x_min) / w
            height = (y_max - y_min) / h

            f.write(f"{cls} {x_center} {y_center} {width} {height} " + " ".join([str(x) for x in soft_label[:-1]]) + "\n") # add soft label entries to the right

In [4]:
train_data = json.load(open("../../our_datasets/soft_PascalVOC_2012_detection_train.json"))

filenames = os.listdir(train_label_path.replace("labels", "images"))
fname_to_img_dimensions = {img["file_name"]: (img["width"], img["height"]) for img in train_data["images"]}

# Create YOLO annotation files for training set
for filename in filenames:

    if not filename in train_data["objects"]:
        # txt_filename = os.path.splitext(filename)[0] + ".txt"
        # txt_path = os.path.join(train_label_path, txt_filename)
        # open(txt_path, "w").close() # create empty txt file for images without annotations

        # txt_path = os.path.join(train_soft_label_path, txt_filename)
        # open(txt_path, "w").close() # create empty txt file for soft labels

        continue

    obj = train_data["objects"][filename] # annotations for this image (list of dicts)

    # create .txt file for this image
    txt_filename = os.path.splitext(filename)[0] + ".txt"
    txt_path = os.path.join(train_label_path, txt_filename)
    with open(txt_path, "w") as f:
        for ann in obj:
            soft_label = ann["postprocessed_soft_label"]
            if max(soft_label[:-1]) < soft_label_thresh:
                continue  # skip if no class has a soft label prob. above the threshold
            
            cls = soft_label.index(max(soft_label[:-1]))  # class with highest soft label probability (exclude background)

            w, h = fname_to_img_dimensions[filename]
            # Convert bbox to YOLO format (x_center, y_center, width, height)
            x_min, y_min, x_max, y_max = ann["bbox"]
            x_center = (x_min + x_max) / 2 / w
            y_center = (y_min + y_max) / 2 / h
            width = (x_max - x_min) / w
            height = (y_max - y_min) / h

            f.write(f"{cls} {x_center} {y_center} {width} {height}\n")

    # store soft label in separate folder 
    txt_path = os.path.join(train_soft_label_path, txt_filename)
    with open(txt_path, "w") as f:
        for ann in obj:
            soft_label = ann["postprocessed_soft_label"]
            if max(soft_label[:-1]) < soft_label_thresh:
                continue  # skip if no class has a soft label prob. above the threshold
            
            cls = soft_label.index(max(soft_label[:-1]))  # class with highest soft label probability (exclude background)

            w, h = fname_to_img_dimensions[filename]
            # Convert bbox to YOLO format (x_center, y_center, width, height)
            x_min, y_min, x_max, y_max = ann["bbox"]
            x_center = (x_min + x_max) / 2 / w
            y_center = (y_min + y_max) / 2 / h
            width = (x_max - x_min) / w
            height = (y_max - y_min) / h

            f.write(f"{cls} {x_center} {y_center} {width} {height} " + " ".join([str(x) for x in soft_label[:-1]]) + "\n") # add soft label entries to the right

# KITTI

In [6]:
import os
import json

existing = "/path/to/datasets/KITTI"
new = "/path/to/datasets/KITTI_ours"

if os.path.exists(new):
    print(f"Removing existing directory: {new}")
    os.system(f"rm -r {new}")

val_label_path = new + "/val" + "/labels"
train_label_path = new + "/train" + "/labels"

val_soft_label_path = val_label_path.replace("labels", "soft_labels")
train_soft_label_path = train_label_path.replace("labels", "soft_labels")

# create new dataset directory structure
os.makedirs(new, exist_ok=True)
os.makedirs(val_label_path, exist_ok=True)
os.makedirs(train_label_path, exist_ok=True)
os.makedirs(val_soft_label_path, exist_ok=True)
os.makedirs(train_soft_label_path, exist_ok=True)
# copy images via symlink
os.system(f"cp -al {existing}/val/images {new}/val/images")
os.system(f"cp -al {existing}/train/images {new}/train/images")

Removing existing directory: /path/to/datasets/KITTI_ours


0

In [7]:
val_data = json.load(open("../../our_datasets/soft_Kitti_2D_train_val.json"))

filenames = list(val_data["objects"].keys())
fname_to_img_dimensions = {img["file_name"]: (img["width"], img["height"]) for img in val_data["images"]}

# Create YOLO annotation files for validation set
for filename in filenames:
    obj = val_data["objects"][filename] # annotations for this image (list of dicts)

    # create .txt file for this image
    txt_filename = os.path.splitext(filename)[0] + ".txt"
    txt_path = os.path.join(val_label_path, txt_filename)
    with open(txt_path, "w") as f:
        for ann in obj:
            soft_label = ann["postprocessed_soft_label"]
            if max(soft_label[:-1]) < soft_label_thresh:
                continue  # skip if no class has a soft label prob. above the threshold
            
            cls = soft_label.index(max(soft_label[:-1]))  # class with highest soft label probability (exclude background)

            w, h = fname_to_img_dimensions[filename + ".png"]
            # Convert bbox to YOLO format (x_center, y_center, width, height)
            x_min, y_min, x_max, y_max = ann["bbox"]
            x_center = (x_min + x_max) / 2 / w
            y_center = (y_min + y_max) / 2 / h
            width = (x_max - x_min) / w
            height = (y_max - y_min) / h

            f.write(f"{cls} {x_center} {y_center} {width} {height}\n")

    # store soft label in separate folder 
    txt_path = os.path.join(val_soft_label_path, txt_filename)
    with open(txt_path, "w") as f:
        for ann in obj:
            soft_label = ann["postprocessed_soft_label"]
            if max(soft_label[:-1]) < soft_label_thresh:
                continue  # skip if no class has a soft label prob. above the threshold
            
            cls = soft_label.index(max(soft_label[:-1]))  # class with highest soft label probability (exclude background)

            w, h = fname_to_img_dimensions[filename + ".png"]
            # Convert bbox to YOLO format (x_center, y_center, width, height)
            x_min, y_min, x_max, y_max = ann["bbox"]
            x_center = (x_min + x_max) / 2 / w
            y_center = (y_min + y_max) / 2 / h
            width = (x_max - x_min) / w
            height = (y_max - y_min) / h

            f.write(f"{cls} {x_center} {y_center} {width} {height} " + " ".join([str(x) for x in soft_label[:-1]]) + "\n") # add soft label entries to the right

In [8]:
train_data = json.load(open("../../our_datasets/soft_Kitti_2D_train_train.json"))

filenames = list(train_data["objects"].keys())
fname_to_img_dimensions = {img["file_name"]: (img["width"], img["height"]) for img in train_data["images"]}

# Create YOLO annotation files for training set
for filename in filenames:
    obj = train_data["objects"][filename] # annotations for this image (list of dicts)

    # create .txt file for this image
    txt_filename = os.path.splitext(filename)[0] + ".txt"
    txt_path = os.path.join(train_label_path, txt_filename)
    with open(txt_path, "w") as f:
        for ann in obj:
            soft_label = ann["postprocessed_soft_label"]
            if max(soft_label[:-1]) < soft_label_thresh:
                continue  # skip if no class has a soft label prob. above the threshold
            
            cls = soft_label.index(max(soft_label[:-1]))  # class with highest soft label probability (exclude background)

            w, h = fname_to_img_dimensions[filename + ".png"]
            # Convert bbox to YOLO format (x_center, y_center, width, height)
            x_min, y_min, x_max, y_max = ann["bbox"]
            x_center = (x_min + x_max) / 2 / w
            y_center = (y_min + y_max) / 2 / h
            width = (x_max - x_min) / w
            height = (y_max - y_min) / h

            f.write(f"{cls} {x_center} {y_center} {width} {height}\n")
    
    # store soft label in separate folder 
    txt_path = os.path.join(train_soft_label_path, txt_filename)
    with open(txt_path, "w") as f:
        for ann in obj:
            soft_label = ann["postprocessed_soft_label"]
            if max(soft_label[:-1]) < soft_label_thresh:
                continue  # skip if no class has a soft label prob. above the threshold
            
            cls = soft_label.index(max(soft_label[:-1]))  # class with highest soft label probability (exclude background)

            w, h = fname_to_img_dimensions[filename + ".png"]
            # Convert bbox to YOLO format (x_center, y_center, width, height)
            x_min, y_min, x_max, y_max = ann["bbox"]
            x_center = (x_min + x_max) / 2 / w
            y_center = (y_min + y_max) / 2 / h
            width = (x_max - x_min) / w
            height = (y_max - y_min) / h

            f.write(f"{cls} {x_center} {y_center} {width} {height} " + " ".join([str(x) for x in soft_label[:-1]]) + "\n") # add soft label entries to the right

# Cityscapes

In [2]:
import os
import json

existing = "/path/to/datasets/Cityscapes"
new = "/path/to/datasets/Cityscapes_ours"

if os.path.exists(new):
    print(f"Removing existing directory: {new}")
    os.system(f"rm -r {new}")

val_label_path = new + "/val" + "/labels"
train_label_path = new + "/train" + "/labels"

val_soft_label_path = val_label_path.replace("labels", "soft_labels")
train_soft_label_path = train_label_path.replace("labels", "soft_labels")

# create new dataset directory structure
os.makedirs(new, exist_ok=True)
os.makedirs(val_label_path, exist_ok=True)
os.makedirs(train_label_path, exist_ok=True)
os.makedirs(val_soft_label_path, exist_ok=True)
os.makedirs(train_soft_label_path, exist_ok=True)
# copy images via symlink
os.system(f"cp -al {existing}/val/images {new}/val/images")
os.system(f"cp -al {existing}/train/images {new}/train/images")

Removing existing directory: /path/to/datasets/Cityscapes_ours


0

In [3]:
val_data = json.load(open("../../our_datasets/soft_Cityscapes_val.json"))

filenames = list(val_data["objects"].keys())
fname_to_img_dimensions = {img["file_name"]: (img["width"], img["height"]) for img in val_data["images"]}

# Create YOLO annotation files for validation set
for filename in filenames:
    obj = val_data["objects"][filename] # annotations for this image (list of dicts)

    # create .txt file for this image
    txt_filename = os.path.splitext(filename)[0] + ".txt"
    txt_path = os.path.join(val_label_path, txt_filename)
    with open(txt_path, "w") as f:
        # here, we sort annotations by class for later code requirements
        for cat in range(8): # iterate over classes in order
            for ann in obj:
                soft_label = ann["postprocessed_soft_label"]
                if max(soft_label[:-1]) < soft_label_thresh:
                    continue  # skip if no class has a soft label prob. above the threshold
                
                cls = soft_label.index(max(soft_label[:-1]))  # class with highest soft label probability (exclude background)

                if cls != cat:
                    continue # skip if this annotation does not belong to the current class -> leads to sorting by class in the final txt file

                w, h = fname_to_img_dimensions[filename + ".png"]
                # Convert bbox to YOLO format (x_center, y_center, width, height)
                x_min, y_min, x_max, y_max = ann["bbox"]
                x_center = (x_min + x_max) / 2 / w
                y_center = (y_min + y_max) / 2 / h
                width = (x_max - x_min) / w
                height = (y_max - y_min) / h

                f.write(f"{cls} {x_center} {y_center} {width} {height}\n")

    # store soft label in separate folder 
    txt_path = os.path.join(val_soft_label_path, txt_filename)
    with open(txt_path, "w") as f:
        for cat in range(8): # iterate over classes in order
            for ann in obj:
                soft_label = ann["postprocessed_soft_label"]
                if max(soft_label[:-1]) < soft_label_thresh:
                    continue  # skip if no class has a soft label prob. above the threshold
                
                cls = soft_label.index(max(soft_label[:-1]))  # class with highest soft label probability (exclude background)
                if cls != cat:
                    continue # skip if this annotation does not belong to the current class -> leads to sorting by class in the final txt file


                w, h = fname_to_img_dimensions[filename + ".png"]
                # Convert bbox to YOLO format (x_center, y_center, width, height)
                x_min, y_min, x_max, y_max = ann["bbox"]
                x_center = (x_min + x_max) / 2 / w
                y_center = (y_min + y_max) / 2 / h
                width = (x_max - x_min) / w
                height = (y_max - y_min) / h

                f.write(f"{cls} {x_center} {y_center} {width} {height} " + " ".join([str(x) for x in soft_label[:-1]]) + "\n") # add soft label entries to the right

In [4]:
from PIL import Image
train_data = json.load(open("../../our_datasets/soft_Cityscapes_train.json"))

filenames = list(train_data["objects"].keys())
# fname_to_img_dimensions = {img["file_name"]: (img["width"], img["height"]) for img in train_data["images"]}
# does not exist yet in the data, so we determine dimensions by loading the image

# Create YOLO annotation files for training set
for filename in filenames:
    obj = train_data["objects"][filename] # annotations for this image (list of dicts)

    # create .txt file for this image
    txt_filename = os.path.splitext(filename)[0] + ".txt"
    txt_path = os.path.join(train_label_path, txt_filename)
    with open(txt_path, "w") as f:
        # here, we sort annotations by class for later code requirements
        for cat in range(8): # iterate over classes in order
            for ann in obj:
                soft_label = ann["postprocessed_soft_label"]
                if max(soft_label[:-1]) < soft_label_thresh:
                    continue  # skip if no class has a soft label prob. above the threshold
                
                cls = soft_label.index(max(soft_label[:-1]))  # class with highest soft label probability (exclude background)

                if cls != cat:
                    continue # skip if this annotation does not belong to the current class -> leads to sorting by class in the final txt file

                img = Image.open(os.path.join(existing, "train/images", filename + ".png"))
                w, h = img.size
                # w, h = fname_to_img_dimensions[filename + ".png"]
                
                # Convert bbox to YOLO format (x_center, y_center, width, height)
                x_min, y_min, x_max, y_max = ann["bbox"]
                x_center = (x_min + x_max) / 2 / w
                y_center = (y_min + y_max) / 2 / h
                width = (x_max - x_min) / w
                height = (y_max - y_min) / h

                f.write(f"{cls} {x_center} {y_center} {width} {height}\n")

    # store soft label in separate folder 
    txt_path = os.path.join(train_soft_label_path, txt_filename)
    with open(txt_path, "w") as f:
        for cat in range(8): # iterate over classes in order
            for ann in obj:
                soft_label = ann["postprocessed_soft_label"]
                
                if max(soft_label[:-1]) < soft_label_thresh:
                    continue  # skip if no class has a soft label prob. above the threshold
                
                cls = soft_label.index(max(soft_label[:-1]))  # class with highest soft label probability (exclude background)
                if cls != cat:
                    continue # skip if this annotation does not belong to the current class -> leads to sorting by class in the final txt file
                img = Image.open(os.path.join(existing, "train/images", filename + ".png"))
                w, h = img.size
                # Convert bbox to YOLO format (x_center, y_center, width, height)
                x_min, y_min, x_max, y_max = ann["bbox"]
                x_center = (x_min + x_max) / 2 / w
                y_center = (y_min + y_max) / 2 / h
                width = (x_max - x_min) / w
                height = (y_max - y_min) / h

                f.write(f"{cls} {x_center} {y_center} {width} {height} " + " ".join([str(x) for x in soft_label[:-1]]) + "\n") # add soft label entries to the right

# COCO

In [2]:
import os
import json

copy_coco_images = True # ! Will create copies of images due to hardlink not working across machines in this case

path_to_val_images = "/path/to/datasets/COCO/2017/val2017"

new = "/path/to/datasets/COCO_ours"

if os.path.exists(new):
    print(f"Removing existing directory: {new}")
    os.system(f"rm -r {new}")

val_label_path = new + "/val" + "/labels"
train_label_path = new + "/train" + "/labels"

val_img_path = val_label_path.replace("labels", "images")
train_img_path = train_label_path.replace("labels", "images")

val_soft_label_path = val_label_path.replace("labels", "soft_labels")
train_soft_label_path = train_label_path.replace("labels", "soft_labels")

# create new dataset directory structure
os.makedirs(new, exist_ok=True)
os.makedirs(val_label_path, exist_ok=True)
os.makedirs(train_label_path, exist_ok=True)
os.makedirs(val_soft_label_path, exist_ok=True)
os.makedirs(train_soft_label_path, exist_ok=True)
os.makedirs(val_img_path, exist_ok=True)
os.makedirs(train_img_path, exist_ok=True)

Removing existing directory: /path/to/datasets/COCO_ours


In [3]:
data = json.load(open("../../our_datasets/soft_COCO_2017_val.json"))
id_to_filename = {img["id"]: img["file_name"] for img in data["images"]}
id_to_img_dimensions = {img["id"]: (img["width"], img["height"]) for img in data["images"]}

In [ ]:
# # write train.txt and val.txt files with image filenames for training and validation splits

# split = json.load(open("../../our_datasets/splits/COCO_train_val_split.json"))

# with open(os.path.join(new, "train.txt"), "w") as train_f, open(os.path.join(new, "val.txt"), "w") as val_f:
#     for id in split["train_imgs"]:
#         filename = id_to_filename[id]
#         train_f.write(f"{filename}\n")
#     for id in split["val_imgs"]:
#         filename = id_to_filename[id]
#         val_f.write(f"{filename}\n")

In [4]:
# copy val and train images
split = json.load(open("../../our_datasets/splits/COCO_train_val_split.json"))
for id in split["val_imgs"]:
    filename = id_to_filename[id]
    os.system(f"cp {path_to_val_images}/{filename} {val_img_path}/")
for id in split["train_imgs"]:
    filename = id_to_filename[id]
    os.system(f"cp {path_to_val_images}/{filename} {train_img_path}/")

In [5]:
val_data = json.load(open("../../our_datasets/soft_COCO_2017_val_val.json"))

filenames = list(val_data["objects"].keys())
fname_to_img_dimensions = {img["file_name"]: (img["width"], img["height"]) for img in val_data["images"]}

# Create YOLO annotation files for validation set
for filename in filenames:
    obj = val_data["objects"][filename] # annotations for this image (list of dicts)

    # create .txt file for this image
    txt_filename = os.path.splitext(filename)[0] + ".txt"
    txt_path = os.path.join(val_label_path, txt_filename)
    with open(txt_path, "w") as f:
        for ann in obj:
            soft_label = ann["postprocessed_soft_label"]
            if max(soft_label[:-1]) < soft_label_thresh:
                continue  # skip if no class has a soft label prob. above the threshold
            
            cls = soft_label.index(max(soft_label[:-1]))  # class with highest soft label probability (exclude background)

            w, h = fname_to_img_dimensions[filename]
            # Convert bbox to YOLO format (x_center, y_center, width, height)
            x_min, y_min, x_max, y_max = ann["bbox"]
            x_center = (x_min + x_max) / 2 / w
            y_center = (y_min + y_max) / 2 / h
            width = (x_max - x_min) / w
            height = (y_max - y_min) / h

            f.write(f"{cls} {x_center} {y_center} {width} {height}\n")

    # store soft label in separate folder 
    txt_path = os.path.join(val_soft_label_path, txt_filename)
    with open(txt_path, "w") as f:
        for ann in obj:
            soft_label = ann["postprocessed_soft_label"]
            if max(soft_label[:-1]) < soft_label_thresh:
                continue  # skip if no class has a soft label prob. above the threshold
            
            cls = soft_label.index(max(soft_label[:-1]))  # class with highest soft label probability (exclude background)

            w, h = fname_to_img_dimensions[filename]
            # Convert bbox to YOLO format (x_center, y_center, width, height)
            x_min, y_min, x_max, y_max = ann["bbox"]
            x_center = (x_min + x_max) / 2 / w
            y_center = (y_min + y_max) / 2 / h
            width = (x_max - x_min) / w
            height = (y_max - y_min) / h

            f.write(f"{cls} {x_center} {y_center} {width} {height} " + " ".join([str(x) for x in soft_label[:-1]]) + "\n") # add soft label entries to the right

In [6]:
train_data = json.load(open("../../our_datasets/soft_COCO_2017_val_train.json"))

filenames = list(train_data["objects"].keys())
fname_to_img_dimensions = {img["file_name"]: (img["width"], img["height"]) for img in train_data["images"]}

# Create YOLO annotation files for training set
for filename in filenames:
    obj = train_data["objects"][filename] # annotations for this image (list of dicts)

    # create .txt file for this image
    txt_filename = os.path.splitext(filename)[0] + ".txt"
    txt_path = os.path.join(train_label_path, txt_filename)
    with open(txt_path, "w") as f:
        for ann in obj:
            soft_label = ann["postprocessed_soft_label"]
            if max(soft_label[:-1]) < soft_label_thresh:
                continue  # skip if no class has a soft label prob. above the threshold
            
            cls = soft_label.index(max(soft_label[:-1]))  # class with highest soft label probability (exclude background)

            w, h = fname_to_img_dimensions[filename]
            # Convert bbox to YOLO format (x_center, y_center, width, height)
            x_min, y_min, x_max, y_max = ann["bbox"]
            x_center = (x_min + x_max) / 2 / w
            y_center = (y_min + y_max) / 2 / h
            width = (x_max - x_min) / w
            height = (y_max - y_min) / h

            f.write(f"{cls} {x_center} {y_center} {width} {height}\n")

    # store soft label in separate folder 
    txt_path = os.path.join(train_soft_label_path, txt_filename)
    with open(txt_path, "w") as f:
        for ann in obj:
            soft_label = ann["postprocessed_soft_label"]
            if max(soft_label[:-1]) < soft_label_thresh:
                continue  # skip if no class has a soft label prob. above the threshold
            
            cls = soft_label.index(max(soft_label[:-1]))  # class with highest soft label probability (exclude background)

            w, h = fname_to_img_dimensions[filename]
            # Convert bbox to YOLO format (x_center, y_center, width, height)
            x_min, y_min, x_max, y_max = ann["bbox"]
            x_center = (x_min + x_max) / 2 / w
            y_center = (y_min + y_max) / 2 / h
            width = (x_max - x_min) / w
            height = (y_max - y_min) / h

            f.write(f"{cls} {x_center} {y_center} {width} {height} " + " ".join([str(x) for x in soft_label[:-1]]) + "\n") # add soft label entries to the right